In [132]:

# Configuração

import pandas as pd
import numpy as np
import json
import warnings
import os

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")



## 1. Carregamento dos Dados Brutos

Fontes:
- **ERP** (`pedido_erp.csv`) — Pedidos da loja física
- **E-commerce** (`pedido_ecom.json`) — Pedidos online
- **CRM** (`clientes_crm.csv`) — Base de clientes

In [133]:

# CÉLULA 2 — Carregamento dos dados brutos

# ERP
erp_raw = pd.read_csv("pedido_erp.csv", sep=";", quotechar='"')

# CRM
crm_raw = pd.read_csv("clientes_crm.csv", sep=";")

# E-commerce (JSON)
with open("pedido_ecom.json", "r", encoding="utf-8") as f:
    ecom_json = json.load(f)

ecom_raw = pd.json_normalize(ecom_json["docs"])

print(f"ERP:        {erp_raw.shape[0]:,} registros | {erp_raw.shape[1]} colunas")
print(f"E-commerce: {ecom_raw.shape[0]:,} registros | {ecom_raw.shape[1]} colunas")
print(f"CRM:        {crm_raw.shape[0]:,} registros | {crm_raw.shape[1]} colunas")

ERP:        11,494 registros | 6 colunas
E-commerce: 1,505 registros | 25 colunas
CRM:        24,520 registros | 11 colunas


## 2. Diagnóstico de Qualidade dos Dados Brutos

Análise de nulos, tipos e anomalias antes do tratamento.

In [134]:

# Diagnóstico de qualidade


def diagnostico(df, nome):
    """Relatório de qualidade de um DataFrame."""
    total = len(df)
    print(f"\n{'='*60}")
    print(f"  DIAGNÓSTICO: {nome} ({total:,} registros)")
    print(f"{'='*60}")
    
    # Nulos
    nulos = df.isnull().sum()
    nulos_pct = (nulos / total * 100).round(1)
    nulos_df = pd.DataFrame({"nulos": nulos, "%": nulos_pct})
    nulos_relevantes = nulos_df[nulos_df["nulos"] > 0].sort_values("%", ascending=False)
    
    if len(nulos_relevantes) > 0:
        print(f"\n  Colunas com nulos:")
        for col, row in nulos_relevantes.iterrows():
            print(f"   {col:<35} {int(row['nulos']):>6} ({row['%']:.1f}%)")
    else:
        print("\n  Nenhuma coluna com nulos")
    
    # Duplicatas — apenas colunas hashable
    try:
        hashable_cols = [c for c in df.columns if df[c].apply(type).isin([list, dict, set]).sum() == 0]
        dupl = df[hashable_cols].duplicated().sum()
        print(f"\n  Duplicatas completas: {dupl:,} ({dupl/total*100:.1f}%)")
    except Exception:
        print(f"\n  Duplicatas: não foi possível calcular (colunas complexas)")
    
    # Tipos
    print(f"\n  Tipos de dados:")
    for dtype, count in df.dtypes.value_counts().items():
        print(f"   {str(dtype):<20} {count} colunas")
    
    # Colunas com listas (JSON nested)
    list_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, (list, dict))).any()]
    if list_cols:
        print(f"\n  Colunas com dados aninhados (list/dict): {list_cols}")
    
    return nulos_relevantes

diag_erp = diagnostico(erp_raw, "ERP")
diag_ecom = diagnostico(ecom_raw, "E-COMMERCE")
diag_crm = diagnostico(crm_raw, "CRM")


  DIAGNÓSTICO: ERP (11,494 registros)

  Nenhuma coluna com nulos

  Duplicatas completas: 0 (0.0%)

  Tipos de dados:
   str                  5 colunas
   int64                1 colunas

  DIAGNÓSTICO: E-COMMERCE (1,505 registros)

  Colunas com nulos:
   summary.coupon.code                   1505 (100.0%)
   summary.coupon.type                   1332 (88.5%)
   settings.source                          3 (0.2%)
   customer.doc                             1 (0.1%)

  Duplicatas completas: 0 (0.0%)

  Tipos de dados:
   float64              9 colunas
   int64                8 colunas
   str                  6 colunas
   object               2 colunas

  Colunas com dados aninhados (list/dict): ['products']

  DIAGNÓSTICO: CRM (24,520 registros)

  Nenhuma coluna com nulos

  Duplicatas completas: 0 (0.0%)

  Tipos de dados:
   str                  9 colunas
   int64                2 colunas


## 3. Tratamento e Padronização

### 3.1 ERP — Pedidos Loja Física

In [135]:
# ============================================================
# CÉLULA 4 — Tratamento ERP
# ============================================================

erp = erp_raw.copy()

# Renomear colunas para padrão português
erp = erp.rename(columns={
    "id": "id_pedido",
    "number": "numero_pedido",
    "customer_document": "documento",
    "seller_name": "vendedor",
    "order_value": "valor_pedido",
    "order_created": "data_pedido"
})

# Converter valor (formato BR "1.234,56" → float)
erp["valor_pedido"] = (
    erp["valor_pedido"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

# Converter data (remover timezone para consistência)
erp["data_pedido"] = pd.to_datetime(erp["data_pedido"], utc=True).dt.tz_localize(None)

# Padronizar vendedor
erp["vendedor"] = erp["vendedor"].str.strip().str.title()

# Padronizar documento (remover espaços)
erp["documento"] = erp["documento"].str.strip()

# Remover dados de teste
mask_teste = erp["documento"] == "99.999.999/9999-99"
print(f"⚠️  Removidos {mask_teste.sum()} pedidos de teste (doc 99.999.999/9999-99)")
erp = erp[~mask_teste].copy()

# Remover duplicatas
dupl = erp.duplicated(subset=["id_pedido"]).sum()
erp = erp.drop_duplicates(subset=["id_pedido"])
print(f"🔄 Removidas {dupl} duplicatas por id_pedido")

# Marcar origem
erp["origem_venda"] = "Loja Física"
erp["canal"] = "Loja"

print(f"\n✅ ERP tratado: {len(erp):,} pedidos")
print(f"   Período: {erp['data_pedido'].min():%Y-%m-%d} a {erp['data_pedido'].max():%Y-%m-%d}")
print(f"   Valor total: R$ {erp['valor_pedido'].sum():,.2f}")
erp.head(3)

⚠️  Removidos 6 pedidos de teste (doc 99.999.999/9999-99)
🔄 Removidas 0 duplicatas por id_pedido

✅ ERP tratado: 11,488 pedidos
   Período: 2016-10-20 a 2024-04-24
   Valor total: R$ 15,862,258.94


,id_pedido,numero_pedido,documento,vendedor,valor_pedido,data_pedido,origem_venda,canal
2,043a2d5c-5b11-43d1-9d91-97ae933ce5b0,8,68.148.678/0001-50,Usuário De Teste,549.00,2016-10-20 13:59:03,Loja Física,Loja
3,a46fe3ad-303a-423f-8e3e-4e59d7bc27e4,9,26.153.919/0001-09,Jessica,"1,328.90",2016-10-27 15:30:19,Loja Física,Loja
4,ffd2fe2c-c49c-4020-8d4c-4c3d49532ba9,10,20.164.518/0001-78,Jessica,"1,418.90",2016-10-31 15:46:47,Loja Física,Loja


### 3.2 E-commerce — Pedidos Online

In [136]:

# Tratamento E-commerce


ecom = ecom_raw.copy()

# Remover TODAS as colunas com dados aninhados (lists/dicts do JSON)
list_cols = [c for c in ecom.columns if ecom[c].apply(lambda x: isinstance(x, (list, dict))).any()]
if list_cols:
    print(f"  Removendo {len(list_cols)} colunas aninhadas: {list_cols}")
    ecom = ecom.drop(columns=list_cols)

# Renomear colunas
rename_map = {
    "_id": "id_pedido_online",
    "orderNumber": "numero_pedido",
    "customer.doc": "documento",
    "settings.source": "canal",
    "settings.createdAt": "data_pedido",
    "summary.order.parts": "quantidade_itens",
    "summary.order.value": "valor_itens",
    "summary.reserved.parts": "quantidade_reservada",
    "summary.reserved.value": "valor_reservado",
    "summary.reserved.rate": "taxa_reserva",
    "summary.promotionalSubtotal": "subtotal_promocional",
    "summary.additionalValues": "valores_adicionais",
    "summary.discount": "desconto",
    "summary.total": "valor_pedido",
    "summary.subtotal": "subtotal",
    "summary.discountRate": "taxa_desconto",
    "summary.invoiceValue": "valor_fatura",
    "summary.subTotalWithFees": "subtotal_com_taxas",
    "seller.name": "vendedor",
    "summary.coupon.type": "tipo_cupom"
}
# Só renomear colunas que existem
rename_map = {k: v for k, v in rename_map.items() if k in ecom.columns}
ecom = ecom.rename(columns=rename_map)

# Converter tipos
ecom["data_pedido"] = pd.to_datetime(ecom["data_pedido"], utc=True).dt.tz_localize(None)
ecom["valor_pedido"] = pd.to_numeric(ecom["valor_pedido"], errors="coerce")

# Preencher nulos com valores semânticos
ecom["tipo_cupom"] = ecom["tipo_cupom"].fillna("Sem Cupom") if "tipo_cupom" in ecom.columns else "Sem Cupom"
ecom["documento"] = ecom["documento"].fillna("nao_identificado")
ecom["canal"] = ecom["canal"].fillna("Não Identificado")

# Padronizar vendedor e canal
ecom["vendedor"] = ecom["vendedor"].str.strip().str.title()
ecom["canal"] = ecom["canal"].str.strip().str.title()

# Padronizar documento
ecom["documento"] = ecom["documento"].str.strip()

# Remover dados de teste
mask_teste = ecom["documento"] == "99.999.999/9999-99"
print(f"  Removidos {mask_teste.sum()} pedidos de teste (E-commerce)")
ecom = ecom[~mask_teste].copy()

# Remover duplicatas por ID
dupl = ecom.duplicated(subset=["id_pedido_online"]).sum()
ecom = ecom.drop_duplicates(subset=["id_pedido_online"])
print(f"  Removidas {dupl} duplicatas por id_pedido_online")

# Marcar origem
ecom["origem_venda"] = "E-commerce"

print(f"\n  E-commerce tratado: {len(ecom):,} pedidos")
print(f"   Período: {ecom['data_pedido'].min():%Y-%m-%d} a {ecom['data_pedido'].max():%Y-%m-%d}")
print(f"   Valor total: R$ {ecom['valor_pedido'].sum():,.2f}")
print(f"   Canais: {ecom['canal'].value_counts().to_dict()}")
ecom.head(3)

  Removendo 1 colunas aninhadas: ['products']
  Removidos 0 pedidos de teste (E-commerce)
  Removidas 0 duplicatas por id_pedido_online

  E-commerce tratado: 1,505 pedidos
   Período: 2023-12-30 a 2024-11-29
   Valor total: R$ 3,008,785.50
   Canais: {'Link': 1009, 'Aplicativo': 278, 'Site': 86, 'Link Sem Preço': 67, 'Vestishop': 53, 'Showroom': 9, 'Não Identificado': 3}


,id_pedido_online,numero_pedido,documento,canal,data_pedido,quantidade_itens,valor_itens,quantidade_reservada,valor_reservado,taxa_reserva,summary.coupon.code,summary.coupon.discount,summary.coupon.amount,summary.regularSubtotal,subtotal_promocional,valores_adicionais,desconto,valor_pedido,subtotal,taxa_desconto,valor_fatura,subtotal_com_taxas,vendedor,tipo_cupom,origem_venda
0,659067b220eab1e0efdc0a37,15000,48.133.237/0001-77,Vestishop,2023-12-30 18:55:46.598,14,"1,378.60",0,0.00,0.00,None,0,0,0.00,0.00,0,0.00,"1,378.60","1,378.60",0.00,0,0,Natalia R.,Sem Cupom,E-commerce
1,6596c98e20eab1e0eff8f927,15001,21.344.407/0001-06,Aplicativo,2024-01-04 15:06:54.932,18,"1,608.20",18,"1,608.20",1.00,None,0,0,0.00,0.00,0,0.00,"1,608.20","1,608.20",0.00,0,0,Fanny,Sem Cupom,E-commerce
2,65985c2720eab1e0ef05c96b,15002,44.348.095/0001-04,Aplicativo,2024-01-05 19:44:39.608,12,848.80,4,449.60,0.53,None,0,0,0.00,0.00,0,0.00,848.80,848.80,0.00,0,0,Natalia R.,Sem Cupom,E-commerce


### 3.3 CRM — Base de Clientes

In [137]:

# Tratamento CRM

crm = crm_raw.copy()

# Renomear colunas
crm = crm.rename(columns={
    "id": "id_cliente",
    "document": "documento",
    "name": "nome_cliente",
    "email": "email",
    "ddi": "ddi",
    "ddd": "ddd",
    "phone": "telefone",
    "buy": "permissao_compra",
    "status": "status_cliente",
    "seller_name": "vendedor_crm",
    "created_at": "data_cadastro"
})

# Converter data
crm["data_cadastro"] = pd.to_datetime(crm["data_cadastro"], utc=True).dt.tz_localize(None)

# Padronizar vendedor — tratar "\\N" como venda direta
crm["vendedor_crm"] = (
    crm["vendedor_crm"]
    .replace("\\N", "Venda Direta")
    .str.strip()
    .str.title()
)

# Padronizar documento
crm["documento"] = crm["documento"].str.strip()

# Padronizar nome do cliente
crm["nome_cliente"] = crm["nome_cliente"].str.strip().str.title()

# Padronizar status e permissão
crm["status_cliente"] = crm["status_cliente"].str.strip().str.lower()
crm["permissao_compra"] = crm["permissao_compra"].str.strip().str.lower()

# Remover dados de teste
mask_teste = crm["documento"] == "99.999.999/9999-99"
print(f"  Removidos {mask_teste.sum()} clientes de teste")
crm = crm[~mask_teste].copy()

# Remover duplicatas por documento (manter cadastro mais recente)
crm = crm.sort_values("data_cadastro", ascending=False)
dupl = crm.duplicated(subset=["documento"]).sum()
crm = crm.drop_duplicates(subset=["documento"], keep="first")
print(f"  Removidas {dupl} duplicatas por documento (mantido mais recente)")

print(f"\n  CRM tratado: {len(crm):,} clientes únicos")
print(f"   Ativos: {(crm['status_cliente'] == 'active').sum():,}")
print(f"   Inativos: {(crm['status_cliente'] == 'inactive').sum():,}")
print(f"   Colunas: {list(crm.columns)}")
crm.head(3)

  Removidos 1 clientes de teste
  Removidas 30 duplicatas por documento (mantido mais recente)

  CRM tratado: 24,489 clientes únicos
   Ativos: 22,645
   Inativos: 1,844
   Colunas: ['id_cliente', 'documento', 'nome_cliente', 'email', 'ddi', 'ddd', 'telefone', 'permissao_compra', 'status_cliente', 'vendedor_crm', 'data_cadastro']


,id_cliente,documento,nome_cliente,email,ddi,ddd,telefone,permissao_compra,status_cliente,vendedor_crm,data_cadastro
13367,ec58ba94-180f-4e6a-bf7f-02a6ba76b7bc,53.100.135/0001-95,Sandilly Kawanny Reis Araujo,email@email.com,55,63,911112222,not allowed,active,Venda Direta,2024-11-29 16:34:58
12438,6b9ff76c-6f0c-4a91-9eca-7c398bb3cb08,16.679.455/0001-15,Rita De Cassia Albano Pires Cerdeira,email@email.com,55,11,911112222,not allowed,active,Venda Direta,2024-11-29 15:44:18
12408,a9b5096c-eb23-411d-a8e4-d71372bc7202,20.651.010/0001-02,Rodrigues &Amp; França,email@email.com,55,89,911112222,allowed,active,Gabi,2024-11-29 15:01:24


## 4. Unificação — Fato Vendas

Concatena ERP + E-commerce em uma tabela única de vendas e enriquece com dados do CRM. Resolve o conflito de `vendedor_x`/`vendedor_y` do merge.

In [138]:

# Unificação: Fato Vendas


# Colunas comuns para concatenação
colunas_vendas = [
    "numero_pedido", "documento", "vendedor", "valor_pedido",
    "data_pedido", "origem_venda", "canal"
]

# Garantir que ambos tenham as mesmas colunas base
erp_union = erp[colunas_vendas].copy()

# E-commerce tem colunas extras — manter separado
ecom_extras = [
    "quantidade_itens", "desconto", "tipo_cupom",
    "subtotal", "valor_fatura"
]
ecom_union = ecom[colunas_vendas + [c for c in ecom_extras if c in ecom.columns]].copy()

# Concatenar
vendas = pd.concat([erp_union, ecom_union], ignore_index=True)

# Merge com CRM — SEM conflito de vendedor
# Selecionar apenas colunas do CRM que interessam
crm_merge = crm[[
    "documento", "id_cliente", "nome_cliente",
    "status_cliente", "permissao_compra", "data_cadastro",
    "vendedor_crm", "ddd"
]].copy()

vendas = vendas.merge(crm_merge, on="documento", how="left")

# Preencher clientes não encontrados no CRM
vendas["nome_cliente"] = vendas["nome_cliente"].fillna("Cliente Não Cadastrado")
vendas["status_cliente"] = vendas["status_cliente"].fillna("desconhecido")

print(f"✅ Fato Vendas unificada: {len(vendas):,} pedidos")
print(f"   Com CRM match: {vendas['id_cliente'].notna().sum():,} ({vendas['id_cliente'].notna().mean()*100:.1f}%)")
print(f"   Sem CRM match: {vendas['id_cliente'].isna().sum():,}")
print(f"\n📋 Colunas: {list(vendas.columns)}")
vendas.head(5)

✅ Fato Vendas unificada: 12,993 pedidos
   Com CRM match: 12,990 (100.0%)
   Sem CRM match: 3

📋 Colunas: ['numero_pedido', 'documento', 'vendedor', 'valor_pedido', 'data_pedido', 'origem_venda', 'canal', 'quantidade_itens', 'desconto', 'tipo_cupom', 'subtotal', 'valor_fatura', 'id_cliente', 'nome_cliente', 'status_cliente', 'permissao_compra', 'data_cadastro', 'vendedor_crm', 'ddd']


,numero_pedido,documento,vendedor,valor_pedido,data_pedido,origem_venda,canal,quantidade_itens,desconto,tipo_cupom,subtotal,valor_fatura,id_cliente,nome_cliente,status_cliente,permissao_compra,data_cadastro,vendedor_crm,ddd
0,8,68.148.678/0001-50,Usuário De Teste,549.00,2016-10-20 13:59:03,Loja Física,Loja,NaN,NaN,NaN,NaN,NaN,08d116e4-58d2-4b4b-9156-ea95a8f34e2c,Guilherme Pj,active,allowed,2016-10-20 13:59:03,Sandra,\N
1,9,26.153.919/0001-09,Jessica,"1,328.90",2016-10-27 15:30:19,Loja Física,Loja,NaN,NaN,NaN,NaN,NaN,43bcca1d-266a-4a55-814e-292da4fee161,Camila Aguimar Arantes,active,allowed,2016-10-27 15:30:19,Jessica,\N
2,10,20.164.518/0001-78,Jessica,"1,418.90",2016-10-31 15:46:47,Loja Física,Loja,NaN,NaN,NaN,NaN,NaN,4fbd3054-4524-49db-b962-fc2e3d1028cb,Gesiela Fernanda Valter,active,allowed,2016-10-31 15:46:47,Jessica,\N
3,11,19.818.860/0001-65,Sandra,129.80,2016-11-01 18:52:20,Loja Física,Loja,NaN,NaN,NaN,NaN,NaN,7fef8954-ee6a-4b73-a0e9-7cc2eeadff21,Rita De Cassia Dos Santos,active,allowed,2016-11-01 18:52:20,Danisio,11
4,12,04.851.514/0001-01,Sandra,892.90,2016-11-02 20:47:14,Loja Física,Loja,NaN,NaN,NaN,NaN,NaN,b0b327a5-bf58-4f60-bc3f-1dfe06df30e0,Maria Lúcia Aragonha Minani Cafelândia-Me,active,allowed,2016-11-02 20:47:14,Sandra,14


## 5. Feature Engineering

Colunas derivadas para o dashboard Power BI e modelo preditivo:
- Dimensões temporais (ano, mês, trimestre, dia da semana)
- Ticket médio, faixas de valor
- Classificação de cliente (novo vs recorrente)
- Métricas RFM (Recência, Frequência, Monetário)

In [139]:

# Feature Engineering: Dimensões Temporais

vendas["ano"] = vendas["data_pedido"].dt.year
vendas["mes"] = vendas["data_pedido"].dt.month
vendas["nome_mes"] = vendas["data_pedido"].dt.strftime("%b")
vendas["trimestre"] = vendas["data_pedido"].dt.quarter
vendas["dia_semana"] = vendas["data_pedido"].dt.day_name()
vendas["ano_mes"] = vendas["data_pedido"].dt.to_period("M").astype(str)
vendas["semana_ano"] = vendas["data_pedido"].dt.isocalendar().week.astype(int)

# Garantir que valor_pedido é numérico (sem reconversão desnecessária)
vendas["valor_pedido"] = pd.to_numeric(vendas["valor_pedido"], errors="coerce")

vendas = vendas[vendas["valor_pedido"] < 100000]

# Faixa de valor do pedido
bins = [0, 200, 500, 1000, 2000, 5000, float("inf")]
labels = ["Até R$200", "R$201-500", "R$501-1000", "R$1001-2000", "R$2001-5000", "Acima R$5000"]
vendas["faixa_valor"] = pd.cut(vendas["valor_pedido"], bins=bins, labels=labels)


print(f"\n📊 Distribuição por faixa de valor:")
print(vendas["faixa_valor"].value_counts().sort_index().to_string())



📊 Distribuição por faixa de valor:
faixa_valor
Até R$200        566
R$201-500       1487
R$501-1000      3669
R$1001-2000     4649
R$2001-5000     2350
Acima R$5000     266


In [140]:
# Feature Engineering: RFM (Recência, Frequência, Monetário)

data_referencia = vendas["data_pedido"].max() + pd.Timedelta(days=1)

rfm = vendas.groupby("documento").agg(
    recencia=("data_pedido", lambda x: (data_referencia - x.max()).days),
    frequencia=("numero_pedido", "nunique"),
    monetario=("valor_pedido", "sum"),
    ticket_medio=("valor_pedido", "mean"),
    primeira_compra=("data_pedido", "min"),
    ultima_compra=("data_pedido", "max"),
    qtd_origens=("origem_venda", "nunique")
).reset_index()

# Classificação RFM via rank percentual (robusto a empates)
rfm["recencia_score"] = pd.cut(
    rfm["recencia"].rank(pct=True, ascending=True),  # menor recência = melhor
    bins=[0, 0.25, 0.5, 0.75, 1.0], labels=[4, 3, 2, 1], include_lowest=True
).astype(int)

rfm["frequencia_score"] = pd.cut(
    rfm["frequencia"].rank(pct=True),
    bins=[0, 0.25, 0.5, 0.75, 1.0], labels=[1, 2, 3, 4], include_lowest=True
).astype(int)

rfm["monetario_score"] = pd.cut(
    rfm["monetario"].rank(pct=True),
    bins=[0, 0.25, 0.5, 0.75, 1.0], labels=[1, 2, 3, 4], include_lowest=True
).astype(int)

rfm["rfm_score"] = rfm["recencia_score"] + rfm["frequencia_score"] + rfm["monetario_score"]

# Segmentação de clientes
def segmentar_cliente(row):
    r, f, m = row["recencia_score"], row["frequencia_score"], row["monetario_score"]
    if r >= 3 and f >= 3 and m >= 3:
        return "VIP"
    elif r >= 3 and f >= 2:
        return "Leal"
    elif r >= 3 and f == 1:
        return "Novo Promissor"
    elif r == 2 and f >= 2:
        return "Em Risco"
    elif r == 1 and f >= 3:
        return "Perdendo VIP"
    elif r == 1:
        return "Inativo"
    else:
        return "Regular"

rfm["segmento"] = rfm.apply(segmentar_cliente, axis=1)

# Cliente omnichannel (compra em mais de um canal)
rfm["omnichannel"] = rfm["qtd_origens"] > 1

print("  Análise RFM calculada")
print(f"\n  Segmentação de clientes:")
print(rfm["segmento"].value_counts().to_string())
print(f"\n  Clientes omnichannel: {rfm['omnichannel'].sum():,} ({rfm['omnichannel'].mean()*100:.1f}%)")
rfm.head(5)

  Análise RFM calculada

  Segmentação de clientes:
segmento
VIP               904
Novo Promissor    750
Inativo           534
Regular           454
Em Risco          442
Perdendo VIP      361
Leal              137

  Clientes omnichannel: 292 (8.2%)


,documento,recencia,frequencia,monetario,ticket_medio,primeira_compra,ultima_compra,qtd_origens,recencia_score,frequencia_score,monetario_score,rfm_score,segmento,omnichannel
0,00.004.449/0001-28,1116,2,"5,765.00","2,882.50",2020-09-28 07:42:09,2021-11-09 23:06:10.000,1,3,3,4,10,VIP,False
1,00.021.784/0001-34,983,1,"1,648.90","1,648.90",2022-03-22 17:11:10,2022-03-22 17:11:10.000,1,3,1,2,6,Novo Promissor,False
2,00.057.859/0001-37,263,2,"9,317.69","4,658.84",2022-02-23 13:58:48,2024-03-11 21:23:01.787,2,4,3,4,11,VIP,True
3,00.070.361/0001-04,1648,1,"1,027.40","1,027.40",2020-05-26 09:30:18,2020-05-26 09:30:18.000,1,2,1,1,4,Regular,False
4,00.086.577/0001-68,1384,5,"7,142.30","1,428.46",2020-10-23 18:21:06,2021-02-14 18:17:38.000,1,2,4,4,10,Em Risco,False


## 6. Séries Temporais — Preparação para Modelo Preditivo

Agrega vendas por mês para alimentar o modelo de previsão do próximo mês.

In [141]:

# Série Temporal Mensal (base para modelo preditivo)

serie_mensal = vendas.groupby("ano_mes").agg(
    receita_total=("valor_pedido", "sum"),
    qtd_pedidos=("numero_pedido", "nunique"),
    clientes_unicos=("documento", "nunique")
).reset_index()

serie_mensal["ticket_medio"] = (
    serie_mensal["receita_total"] /
    serie_mensal["qtd_pedidos"]
)

serie_mensal["ano_mes_dt"] = pd.to_datetime(serie_mensal["ano_mes"])
serie_mensal = serie_mensal.sort_values("ano_mes_dt")

# Variação mês a mês
serie_mensal["var_receita_pct"] = serie_mensal["receita_total"].pct_change() * 100
serie_mensal["var_pedidos_pct"] = serie_mensal["qtd_pedidos"].pct_change() * 100

# Média móvel 3 meses (suavização)
serie_mensal["receita_mm3"] = serie_mensal["receita_total"].rolling(3).mean()
serie_mensal["pedidos_mm3"] = serie_mensal["qtd_pedidos"].rolling(3).mean()

print(f"   Meses: {len(serie_mensal)} ({serie_mensal['ano_mes'].iloc[0]} a {serie_mensal['ano_mes'].iloc[-1]})")
print(f"\n📊 Últimos 6 meses:")
print(serie_mensal[["ano_mes", "receita_total", "qtd_pedidos", "ticket_medio", "clientes_unicos"]].tail(6).to_string(index=False))

   Meses: 96 (2016-10 a 2024-11)

📊 Últimos 6 meses:
ano_mes  receita_total  qtd_pedidos  ticket_medio  clientes_unicos
2024-06     273,444.98          139      1,967.23              108
2024-07     225,816.83           98      2,304.25               86
2024-08     266,970.44          124      2,152.99               97
2024-09     232,904.58          121      1,924.83               98
2024-10     324,263.87          143      2,267.58              109
2024-11     220,153.85          100      2,201.54               80


## 7. Métricas por Vendedor (Persona: Gerente de Loja / Vendedor)

In [142]:
# Performance por Vendedor

perf_vendedor = vendas.groupby("vendedor").agg(
    total_vendas=("valor_pedido", "sum"),
    qtd_pedidos=("numero_pedido", "count"),
    ticket_medio=("valor_pedido", "mean"),
    clientes_atendidos=("documento", "nunique"),
    primeiro_pedido=("data_pedido", "min"),
    ultimo_pedido=("data_pedido", "max")
).reset_index()

perf_vendedor["pedidos_por_cliente"] = (
    perf_vendedor["qtd_pedidos"] / perf_vendedor["clientes_atendidos"]
).round(2)

perf_vendedor = perf_vendedor.sort_values("total_vendas", ascending=False)

print(f"\n📊 Top 10 vendedores por receita:")
print(perf_vendedor[["vendedor", "total_vendas", "qtd_pedidos", "ticket_medio", "clientes_atendidos"]].head(10).to_string(index=False))


📊 Top 10 vendedores por receita:
        vendedor  total_vendas  qtd_pedidos  ticket_medio  clientes_atendidos
            Keli  3,146,486.27         2038      1,543.91                 405
          Denize  3,126,567.79         2054      1,522.18                 404
         Natalia  2,761,515.21         2030      1,360.35                 688
           Fanny  2,394,294.88         1693      1,414.23                 605
           Erika  1,918,510.94         1439      1,333.23                 424
      Natalia R.  1,734,279.80         1094      1,585.26                 516
            Gabi  1,221,194.07          820      1,489.26                 346
         Danisio  1,073,525.14          666      1,611.90                 240
          Sandra  1,004,020.26          739      1,358.62                 451
Usuário De Teste    114,726.09           73      1,571.59                  45


## 8. Exportação para Power BI

Exporta tabelas tratadas em formato CSV para importação no Power BI.

**Modelo Star Schema:**
- `fato_vendas.csv` — Tabela fato (cada pedido)
- `dim_clientes.csv` — Dimensão clientes (com RFM e segmento)
- `dim_vendedores.csv` — Dimensão vendedores (performance)
- `serie_temporal.csv` — Agregação mensal (para modelo preditivo)

In [143]:

# Exportação para Power BI

output_dir = "data_powerbi"
os.makedirs(output_dir, exist_ok=True)

# Arredondar colunas monetárias antes de exportar
vendas["valor_pedido"] = vendas["valor_pedido"].round(2)

# 1. Fato Vendas
vendas.to_csv(f"{output_dir}/fato_vendas.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")

# 2. Dimensão Clientes (com RFM)
dim_clientes = crm.merge(
    rfm[["documento", "recencia", "frequencia", "monetario", "ticket_medio",
         "segmento", "omnichannel", "primeira_compra", "ultima_compra"]],
    on="documento",
    how="left"
)
dim_clientes["monetario"] = dim_clientes["monetario"].round(2)
dim_clientes["ticket_medio"] = dim_clientes["ticket_medio"].round(2)
dim_clientes.to_csv(f"{output_dir}/dim_clientes.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")

# 3. Dimensão Vendedores
perf_vendedor["total_vendas"] = perf_vendedor["total_vendas"].round(2)
perf_vendedor["ticket_medio"] = perf_vendedor["ticket_medio"].round(2)
perf_vendedor.to_csv(f"{output_dir}/dim_vendedores.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")

# 4. Série Temporal
serie_mensal["receita_total"] = serie_mensal["receita_total"].round(2)
serie_mensal["ticket_medio"] = serie_mensal["ticket_medio"].round(2)
serie_mensal["var_receita_pct"] = serie_mensal["var_receita_pct"].round(2)
serie_mensal["var_pedidos_pct"] = serie_mensal["var_pedidos_pct"].round(2)
serie_mensal["receita_mm3"] = serie_mensal["receita_mm3"].round(2)
serie_mensal["pedidos_mm3"] = serie_mensal["pedidos_mm3"].round(2)
serie_mensal.to_csv(f"{output_dir}/serie_temporal.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")

print(f"\n📁 Todos os arquivos exportados em '{output_dir}/' (formato BR: decimal com vírgula)")



📁 Todos os arquivos exportados em 'data_powerbi/' (formato BR: decimal com vírgula)


## 9. Resumo Final — Validação do Tratamento

In [144]:

# Resumo Final

print("=" * 65)
print("         RESUMO DO TRATAMENTO DE DADOS — PROJETO VESTI")
print("=" * 65)

print(f"""
📦 DADOS BRUTOS:
   ERP:        {len(erp_raw):>6,} registros
   E-commerce: {len(ecom_raw):>6,} registros
   CRM:        {len(crm_raw):>6,} registros

🔧 APÓS TRATAMENTO:
   ERP:        {len(erp):>6,} pedidos (loja física)
   E-commerce: {len(ecom):>6,} pedidos (online)
   CRM:        {len(crm):>6,} clientes únicos
   
📊 FATO VENDAS UNIFICADA:
   Total:      {len(vendas):>6,} pedidos
   Período:    {vendas['data_pedido'].min():%Y-%m-%d} a {vendas['data_pedido'].max():%Y-%m-%d}
   Receita:    R$ {vendas['valor_pedido'].sum():>12,.2f}
   Ticket Médio: R$ {vendas['valor_pedido'].mean():>8,.2f}

🎯 SEGMENTAÇÃO RFM:
{rfm['segmento'].value_counts().to_string()}

📁 ARQUIVOS EXPORTADOS (Power BI):
   data_powerbi/fato_vendas.csv
   data_powerbi/dim_clientes.csv
   data_powerbi/dim_vendedores.csv
   data_powerbi/serie_temporal.csv
""")



         RESUMO DO TRATAMENTO DE DADOS — PROJETO VESTI

📦 DADOS BRUTOS:
   ERP:        11,494 registros
   E-commerce:  1,505 registros
   CRM:        24,520 registros

🔧 APÓS TRATAMENTO:
   ERP:        11,488 pedidos (loja física)
   E-commerce:  1,505 pedidos (online)
   CRM:        24,489 clientes únicos

📊 FATO VENDAS UNIFICADA:
   Total:      12,993 pedidos
   Período:    2016-10-20 a 2024-11-29
   Receita:    R$ 18,871,044.44
   Ticket Médio: R$ 1,452.40

🎯 SEGMENTAÇÃO RFM:
segmento
VIP               904
Novo Promissor    750
Inativo           534
Regular           454
Em Risco          442
Perdendo VIP      361
Leal              137

📁 ARQUIVOS EXPORTADOS (Power BI):
   data_powerbi/fato_vendas.csv
   data_powerbi/dim_clientes.csv
   data_powerbi/dim_vendedores.csv
   data_powerbi/serie_temporal.csv



---

## 10. Modelo Preditivo — Previsão de Receita e Pedidos (Mês Seguinte)

**Abordagem:** Comparação entre 3 modelos para escolher o melhor:
1. **Holt-Winters (ETS)** — Suavização exponencial com sazonalidade
2. **SARIMA** — Modelo autorregressivo sazonal
3. **Regressão com features sazonais** — Baseline linear

**Métricas de avaliação:** MAE, MAPE, RMSE nos últimos 6 meses (validação)

In [145]:
# Preparação da série temporal para modelagem

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Série temporal indexada por data
ts = serie_mensal[["ano_mes_dt", "receita_total", "qtd_pedidos", "ticket_medio", "clientes_unicos"]].copy()
ts = ts.set_index("ano_mes_dt")
ts.index = pd.DatetimeIndex(ts.index).to_period("M").to_timestamp()
ts = ts.asfreq("MS")

# Preencher meses faltantes (se houver gaps)
ts = ts.ffill()

# Split treino/validação (últimos 6 meses para teste)
n_test = 6
train = ts.iloc[:-n_test]
test = ts.iloc[-n_test:]

print(f"  Série temporal preparada")
print(f"   Total: {len(ts)} meses")
print(f"   Treino: {len(train)} meses ({train.index[0]:%Y-%m} a {train.index[-1]:%Y-%m})")
print(f"   Teste:  {len(test)} meses ({test.index[0]:%Y-%m} a {test.index[-1]:%Y-%m})")
print(f"\n  Estatísticas da receita mensal (treino):")
print(f"   Média:  R$ {train['receita_total'].mean():,.2f}")
print(f"   Desvio: R$ {train['receita_total'].std():,.2f}")
print(f"   Min:    R$ {train['receita_total'].min():,.2f}")
print(f"   Max:    R$ {train['receita_total'].max():,.2f}")

  Série temporal preparada
   Total: 98 meses
   Treino: 92 meses (2016-10 a 2024-05)
   Teste:  6 meses (2024-06 a 2024-11)

  Estatísticas da receita mensal (treino):
   Média:  R$ 188,432.42
   Desvio: R$ 132,280.61
   Min:    R$ 626.50
   Max:    R$ 625,763.00


In [146]:
# CÉLULA 15 — Modelo 1: Holt-Winters (Exponential Smoothing)
def avaliar_modelo(nome, real, previsto):
    """Calcula métricas de avaliação."""
    mae = mean_absolute_error(real, previsto)
    rmse = np.sqrt(mean_squared_error(real, previsto))
    mape = np.mean(np.abs((real - previsto) / real)) * 100
    return {"modelo": nome, "MAE": mae, "RMSE": rmse, "MAPE_%": mape}

# Holt-Winters com sazonalidade multiplicativa (12 meses)
hw_model = ExponentialSmoothing(
    train["receita_total"],
    trend="add",
    seasonal="mul",
    seasonal_periods=12
).fit(optimized=True)

hw_pred = hw_model.forecast(n_test)
hw_metrics = avaliar_modelo("Holt-Winters", test["receita_total"].values, hw_pred.values)

print("  Modelo 1: Holt-Winters (ETS)")
print(f"   MAE:  R$ {hw_metrics['MAE']:,.2f}")
print(f"   RMSE: R$ {hw_metrics['RMSE']:,.2f}")
print(f"   MAPE: {hw_metrics['MAPE_%']:.1f}%")
print(f"\n  Previsões vs Real (últimos 6 meses):")
comparacao_hw = pd.DataFrame({
    "mes": test.index.strftime("%Y-%m"),
    "real": test["receita_total"].values,
    "previsto": hw_pred.values,
    "erro_%": ((hw_pred.values - test["receita_total"].values) / test["receita_total"].values * 100)
})
print(comparacao_hw.to_string(index=False))

  Modelo 1: Holt-Winters (ETS)
   MAE:  R$ 48,503.71
   RMSE: R$ 57,495.28
   MAPE: 18.1%

  Previsões vs Real (últimos 6 meses):
    mes       real   previsto  erro_%
2024-06 273,444.98 255,906.41   -6.41
2024-07 225,816.83 176,982.73  -21.63
2024-08 266,970.44 206,672.04  -22.59
2024-09 232,904.58 210,177.36   -9.76
2024-10 324,263.87 214,976.07  -33.70
2024-11 220,153.85 187,817.71  -14.69


In [147]:
# Modelo 2: SARIMA

# SARIMA(1,1,1)(1,1,1,12) — configuração clássica para dados mensais com sazonalidade anual
sarima_model = SARIMAX(
    train["receita_total"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

sarima_pred = sarima_model.forecast(n_test)
sarima_metrics = avaliar_modelo("SARIMA", test["receita_total"].values, sarima_pred.values)

print("  Modelo 2: SARIMA(1,1,1)(1,1,1,12)")
print(f"   MAE:  R$ {sarima_metrics['MAE']:,.2f}")
print(f"   RMSE: R$ {sarima_metrics['RMSE']:,.2f}")
print(f"   MAPE: {sarima_metrics['MAPE_%']:.1f}%")
print(f"\n  Previsões vs Real:")
comparacao_sarima = pd.DataFrame({
    "mes": test.index.strftime("%Y-%m"),
    "real": test["receita_total"].values,
    "previsto": sarima_pred.values,
    "erro_%": ((sarima_pred.values - test["receita_total"].values) / test["receita_total"].values * 100)
})
print(comparacao_sarima.to_string(index=False))

  Modelo 2: SARIMA(1,1,1)(1,1,1,12)
   MAE:  R$ 30,603.63
   RMSE: R$ 45,132.37
   MAPE: 10.5%

  Previsões vs Real:
    mes       real   previsto  erro_%
2024-06 273,444.98 242,478.87  -11.32
2024-07 225,816.83 220,173.81   -2.50
2024-08 266,970.44 227,168.03  -14.91
2024-09 232,904.58 226,426.07   -2.78
2024-10 324,263.87 226,299.71  -30.21
2024-11 220,153.85 217,386.28   -1.26


In [148]:
# Modelo 3: Regressão Linear com Features Sazonais

def criar_features_temporais(df):
    """Cria features para regressão a partir do índice datetime."""
    feat = pd.DataFrame(index=df.index)
    feat["trend"] = np.arange(len(df))
    feat["mes"] = df.index.month
    # Encodar meses como sin/cos (captura ciclicidade)
    feat["mes_sin"] = np.sin(2 * np.pi * feat["mes"] / 12)
    feat["mes_cos"] = np.cos(2 * np.pi * feat["mes"] / 12)
    feat["trimestre"] = df.index.quarter
    return feat[["trend", "mes_sin", "mes_cos", "trimestre"]]

X_train = criar_features_temporais(train)
X_test = criar_features_temporais(test)
# Ajustar trend do teste para continuidade
X_test["trend"] = np.arange(len(train), len(train) + len(test))

lr_model = LinearRegression().fit(X_train, train["receita_total"])
lr_pred = lr_model.predict(X_test)
lr_metrics = avaliar_modelo("Regressão Linear", test["receita_total"].values, lr_pred)

print("  Modelo 3: Regressão Linear Sazonal")
print(f"   MAE:  R$ {lr_metrics['MAE']:,.2f}")
print(f"   RMSE: R$ {lr_metrics['RMSE']:,.2f}")
print(f"   MAPE: {lr_metrics['MAPE_%']:.1f}%")

  Modelo 3: Regressão Linear Sazonal
   MAE:  R$ 39,076.39
   RMSE: R$ 56,442.54
   MAPE: 14.2%


In [149]:
# Comparação dos Modelos e Seleção do Melhor

resultados = pd.DataFrame([hw_metrics, sarima_metrics, lr_metrics])
resultados = resultados.sort_values("MAPE_%")

print("=" * 65)
print("      COMPARAÇÃO DOS MODELOS PREDITIVOS")
print("=" * 65)
print(f"\n{'Modelo':<25} {'MAE (R$)':>12} {'RMSE (R$)':>12} {'MAPE (%)':>10}")
print("-" * 60)
for _, r in resultados.iterrows():
    marcador = " <-- MELHOR" if r.name == resultados.index[0] else ""
    print(f"{r['modelo']:<25} {r['MAE']:>12,.2f} {r['RMSE']:>12,.2f} {r['MAPE_%']:>9.1f}%{marcador}")

melhor_modelo = resultados.iloc[0]["modelo"]
print(f"\n  Modelo selecionado: {melhor_modelo} (menor MAPE)")

      COMPARAÇÃO DOS MODELOS PREDITIVOS

Modelo                        MAE (R$)    RMSE (R$)   MAPE (%)
------------------------------------------------------------
SARIMA                       30,603.63    45,132.37      10.5% <-- MELHOR
Regressão Linear             39,076.39    56,442.54      14.2%
Holt-Winters                 48,503.71    57,495.28      18.1%

  Modelo selecionado: SARIMA (menor MAPE)


## 11. Previsão para o Próximo Mês (Dezembro 2024)

Retreina o melhor modelo com todos os dados e gera previsão para os próximos 3 meses, com intervalo de confiança.

In [150]:
# Previsão Final com SARIMA (melhor modelo)

n_forecast = 3  # Próximos 3 meses

# Retreinar SARIMA com todos os dados
sarima_final = SARIMAX(
    ts["receita_total"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

forecast_receita = sarima_final.forecast(n_forecast)

# SARIMA para pedidos
sarima_pedidos = SARIMAX(
    ts["qtd_pedidos"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

forecast_pedidos = sarima_pedidos.forecast(n_forecast)

# SARIMA para clientes
sarima_clientes = SARIMAX(
    ts["clientes_unicos"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

forecast_clientes = sarima_clientes.forecast(n_forecast)

# Intervalo de confiança baseado no MAPE da validação
mape_modelo = sarima_metrics["MAPE_%"] / 100
margem = forecast_receita * mape_modelo

# Montar tabela de previsão
previsao = pd.DataFrame({
    "mes": forecast_receita.index.strftime("%Y-%m"),
    "receita_prevista": forecast_receita.values,
    "receita_min": (forecast_receita - margem).values,
    "receita_max": (forecast_receita + margem).values,
    "pedidos_previstos": forecast_pedidos.values.astype(int),
    "clientes_previstos": forecast_clientes.values.astype(int),
    "ticket_medio_previsto": (forecast_receita / forecast_pedidos).values
})

print("=" * 65)
print("   PREVISÃO SARIMA — PRÓXIMOS 3 MESES")
print("=" * 65)
for _, row in previsao.iterrows():
    print(f"""
  {row['mes']}:
   Receita:  R$ {row['receita_prevista']:>12,.2f}  (R$ {row['receita_min']:,.0f} ~ R$ {row['receita_max']:,.0f})
   Pedidos:  {row['pedidos_previstos']:>6}
   Clientes: {row['clientes_previstos']:>6}
   Ticket:   R$ {row['ticket_medio_previsto']:>8,.2f}""")

# Comparar com mesmo mês do ano anterior (onde disponível)
print(f"\n  Comparação com mesmo período do ano anterior:")
for _, row in previsao.iterrows():
    mes_num = int(row["mes"].split("-")[1])
    mesmo_mes_anterior = ts[(ts.index.month == mes_num) & (ts.index.year == 2023)]
    if len(mesmo_mes_anterior) > 0 and mesmo_mes_anterior.iloc[-1]["receita_total"] > 10000:
        val_anterior = mesmo_mes_anterior.iloc[-1]["receita_total"]
        var = (row["receita_prevista"] - val_anterior) / val_anterior * 100
        print(f"   {row['mes']} vs {2023}-{mes_num:02d}: {var:+.1f}% (R$ {val_anterior:,.0f} -> R$ {row['receita_prevista']:,.0f})")

   PREVISÃO SARIMA — PRÓXIMOS 3 MESES

  2024-12:
   Receita:  R$   201,387.53  (R$ 180,248 ~ R$ 222,527)
   Pedidos:      57
   Clientes:     52
   Ticket:   R$ 3,509.99

  2025-01:
   Receita:  R$   191,315.07  (R$ 171,233 ~ R$ 211,397)
   Pedidos:      81
   Clientes:     71
   Ticket:   R$ 2,347.80

  2025-02:
   Receita:  R$   209,213.93  (R$ 187,253 ~ R$ 231,175)
   Pedidos:     126
   Clientes:     92
   Ticket:   R$ 1,657.76

  Comparação com mesmo período do ano anterior:
   2025-01 vs 2023-01: +52.0% (R$ 125,905 -> R$ 191,315)
   2025-02 vs 2023-02: +52.6% (R$ 137,079 -> R$ 209,214)


In [151]:

# Exportar previsões para Power BI

# Arredondar previsões para 2 casas decimais
previsao["receita_prevista"] = previsao["receita_prevista"].round(2)
previsao["receita_min"] = previsao["receita_min"].round(2)
previsao["receita_max"] = previsao["receita_max"].round(2)
previsao["ticket_medio_previsto"] = previsao["ticket_medio_previsto"].round(2)

# Previsões futuras
previsao.to_csv(f"{output_dir}/previsao_proximos_meses.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
print(f"  {output_dir}/previsao_proximos_meses.csv exportado")

# Série histórica + previsão combinada (para gráfico no Power BI)
historico_previsao = serie_mensal[["ano_mes", "ano_mes_dt", "receita_total", "qtd_pedidos", "ticket_medio", "clientes_unicos"]].copy()
historico_previsao["tipo"] = "Realizado"

# Ponto de conexão: último mês realizado também entra como Previsto
# para que as duas linhas se conectem visualmente no gráfico
ultimo_realizado = historico_previsao.iloc[[-1]].copy()
ultimo_realizado["tipo"] = "Previsto"

futuro = previsao[["mes", "receita_prevista", "pedidos_previstos", "ticket_medio_previsto", "clientes_previstos"]].copy()
futuro.columns = ["ano_mes", "receita_total", "qtd_pedidos", "ticket_medio", "clientes_unicos"]
futuro["ano_mes_dt"] = pd.to_datetime(futuro["ano_mes"])
futuro["tipo"] = "Previsto"

# Garantir mesma ordem de colunas
cols = ["ano_mes", "ano_mes_dt", "receita_total", "qtd_pedidos", "ticket_medio", "clientes_unicos", "tipo"]
futuro = futuro[cols]

serie_completa = pd.concat([historico_previsao, ultimo_realizado, futuro], ignore_index=True)
serie_completa["receita_total"] = serie_completa["receita_total"].round(2)
serie_completa["ticket_medio"] = serie_completa["ticket_medio"].round(2)
serie_completa.to_csv(f"{output_dir}/serie_historico_previsao.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
print(f"  {output_dir}/serie_historico_previsao.csv exportado ({len(serie_completa)} registros)")

# Resultados dos modelos
resultados.to_csv(f"{output_dir}/comparacao_modelos.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
print(f"  {output_dir}/comparacao_modelos.csv exportado")

print(f"\n  Todos os arquivos de previsão exportados em '{output_dir}/' (formato BR: decimal com vírgula)")


  data_powerbi/previsao_proximos_meses.csv exportado
  data_powerbi/serie_historico_previsao.csv exportado (100 registros)
  data_powerbi/comparacao_modelos.csv exportado

  Todos os arquivos de previsão exportados em 'data_powerbi/' (formato BR: decimal com vírgula)
